# scramp

## Example 1: a full SCRAM-SHA-256 login, client against server

In [ ]:
# Example 1: the four-message exchange (client-first, server-first, client-final, server-final) succeeds
from scramp import ScramClient, ScramMechanism

mechanism = ScramMechanism()  # SCRAM-SHA-256 by default
db = {"user": mechanism.make_auth_info("pencil")}  # salt, stored key, server key, iteration count
server = mechanism.make_server(lambda username: db[username])
client = ScramClient(["SCRAM-SHA-256"], "user", "pencil")
server.set_client_first(client.get_client_first())
client.set_server_first(server.get_server_first())
server.set_client_final(client.get_client_final())
client.set_server_final(server.get_server_final())  # raises if the server's proof is wrong
print("Example 1: login succeeded with", mechanism.name, "and", db["user"][3], "iterations")

## Example 2: a wrong password is refused by the server

In [ ]:
# Example 2: the server rejects the client's proof when the password does not match
from scramp import ScramClient, ScramException, ScramMechanism

mechanism = ScramMechanism()
db = {"user": mechanism.make_auth_info("pencil")}
server = mechanism.make_server(lambda username: db[username])
client = ScramClient(["SCRAM-SHA-256"], "user", "not-the-password")
server.set_client_first(client.get_client_first())
client.set_server_first(server.get_server_first())
try:
    server.set_client_final(client.get_client_final())
except ScramException as error:
    print("Example 2: wrong password refused:", str(error)[:80])
else:
    raise AssertionError("the server accepted a wrong password")

## Example 3: the RFC 7677 test vector, client side

In [ ]:
# Example 3: with a fixed client nonce the client's messages match the SCRAM-SHA-256 example of RFC 7677
from scramp import ScramClient

client = ScramClient(["SCRAM-SHA-256"], "user", "pencil", c_nonce="rOprNGfwEbeRWgbNEkqO")
assert client.get_client_first() == "n,,n=user,r=rOprNGfwEbeRWgbNEkqO"
client.set_server_first("r=rOprNGfwEbeRWgbNEkqO%hvYDpWUa2RaTCAfuxFIlj)hNlF$k0,s=W22ZaJ0SNY7soEsUEjb6gQ==,i=4096")
final = client.get_client_final()
assert final.endswith("p=dHzbZapWIk4jUhN+Ute9ytag9zjfMHgsqmmiz7AndVQ="), final
client.set_server_final("v=6rriTRBi23WpRR/wtup+mMhUZUn/dB5nLTJRsjl95G4=")
print("Example 3: RFC 7677 client proof reproduced:", final.rsplit(",", 1)[-1])

## Example 4: the client picks the strongest mechanism both sides offer

In [ ]:
# Example 4: offered SHA-1 and SHA-512, the client chooses SCRAM-SHA-512; the server agrees on it
from scramp import ScramClient, ScramMechanism

client = ScramClient(["SCRAM-SHA-1", "SCRAM-SHA-512"], "user", "pencil")
mechanism = ScramMechanism("SCRAM-SHA-512")
db = {"user": mechanism.make_auth_info("pencil")}
server = mechanism.make_server(lambda username: db[username])
server.set_client_first(client.get_client_first())
client.set_server_first(server.get_server_first())
server.set_client_final(client.get_client_final())
client.set_server_final(server.get_server_final())
print("Example 4: negotiated", client.mechanism_name, "and logged in")

## Example 5: channel binding (SCRAM-SHA-256-PLUS) with tls-unique data

In [ ]:
# Example 5: both sides bind the login to the same TLS channel data; mismatched data is refused
from scramp import ScramClient, ScramException, ScramMechanism

def login(client_cb, server_cb):
    mech = ScramMechanism("SCRAM-SHA-256-PLUS")
    db = {"user": mech.make_auth_info("pencil")}
    server = mech.make_server(lambda u: db[u], channel_binding=server_cb)
    client = ScramClient(["SCRAM-SHA-256-PLUS"], "user", "pencil", channel_binding=client_cb)
    server.set_client_first(client.get_client_first())
    client.set_server_first(server.get_server_first())
    server.set_client_final(client.get_client_final())
    client.set_server_final(server.get_server_final())

login(("tls-unique", b"channel-A"), ("tls-unique", b"channel-A"))
try:
    login(("tls-unique", b"channel-A"), ("tls-unique", b"channel-B"))
except ScramException as error:
    print("Example 5: bound login works; a different channel is refused:", str(error)[:60])
else:
    raise AssertionError("a login bound to another channel was accepted")